# Capstone — Research Paper Analysis & Decision-Support System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShubhamSnSharma/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

### Title: Predicting Organic Search Traffic Decay in Enterprise Content Portfolios: A Machine Learning Approach to Decision-Support Prioritization

**Author:** Shubham Sharma  
**Track:** Machine Learning · FlyRank AI ML Internship  
**Dataset:** FlyRank Internship Warehouse (`fact_content_daily_performance`, 79M+ rows, v20260703 build)  
**Repository:** [github.com/ShubhamSnSharma/flyrank-ml-internship](https://github.com/ShubhamSnSharma/flyrank-ml-internship)

---

### Abstract (5-Sentence Summary)
1. **Question**: Can machine learning models help prioritize content pages vulnerable to future organic search traffic decay to guide scarce human editorial review bandwidth?  
2. **Data**: The warehouse contains 79M+ rows of production search-performance data; after applying the study's eligibility criteria, the analysis cohort contained 16,513 eligible pages across 36 clients spanning February 1 through May 31, 2026.  
3. **Methodology**: We train a standardized Logistic Regression classifier using 9 pre-May historical/static search and engagement features under a strict client-grouped holdout validation design to predict a May 1–31, 2026 forward evaluation window organic click decline of $\ge 20\%$.  
4. **Results**: On the primary held-out client split, the model achieved a Precision@50 of 0.2400 (12/50 true positives) compared to 0.2000 (10/50) for a 3-signal heuristic baseline (+4.0 percentage points), while repeated 5-split validation demonstrated that performance lift is partition-sensitive (mean lift: +1.6 percentage points, range: -12.0 to +14.0 pp).  
5. **Decision Support**: We translate model predictions into a decision-support Content Action Playbook with 5 illustrative review archetypes, explainable reason codes, and human-in-the-loop safety guardrails that explicitly prohibit autonomous CMS modifications.

## 1. Question & Problem Framing

### The Operational Decision Problem
In modern enterprise digital publishing, content portfolios frequently scale to tens of thousands of published URLs. Over time, search algorithm updates, evolving user intent, and competitor content releases can contribute to changes in organic search traffic. However, editorial teams operate under finite diagnostic bandwidth — with 15–20 high-depth reviews per week serving as an illustrative editorial planning assumption.

Without a ranking model, editorial teams may need to rely on manual sorting or simple heuristics, which can make it difficult to allocate limited review capacity consistently.

### Unit of Analysis & Target Decision
- **Unit of Analysis**: An individual published URL (`content_hash_id`) within a client portfolio (`client_hash_id`) over the forward evaluation window.
- **Target Decision**: Prioritizing pages for human diagnostic review, with the top 50 candidates used as the primary evaluation queue.
- **Cost of Errors**: False positives can allocate illustrative editorial review effort (~2.0–5.0h planning assumption per page) to pages that do not meet the study's decline label; false negatives can give lower review priority to pages that subsequently meet the decline label.

> **Core Playbook Philosophy**: The model supplies a prioritization signal, not a diagnosis:
> $$\text{Model signal} \longrightarrow \text{Review priority} \longrightarrow \text{Human diagnosis} \longrightarrow \text{Editorial decision}$$

In [ ]:
import os, json, duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from getpass import getpass
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

print("=== ML-11 Capstone: Environment & Dependencies Initialized ===")

## 2. Data Source, Cohort Filtering & Safety

### Warehouse Release & Table Structure
We query FlyRank's hosted warehouse release (`hf://datasets/FlyRank/internship-warehouse`, build v20260703) using DuckDB. The warehouse contains 79M+ rows of production search-performance data. After the study's eligibility criteria were applied, the analysis cohort contained 16,513 eligible pages across 36 clients.

### Temporal Windows & Data Split
- **Feature Window**: February 1, 2026 – April 30, 2026 (pre-May historical aggregation).
- **Target Window**: May 1–31, 2026 forward evaluation window (strictly isolated forward outcome period).
- **Zero Temporal Overlap**: All 9 production features are strictly constructed from pre-May data.

### Cohort Inclusion Criteria
To filter out newly launched or unindexed pages with volatile search noise, we apply standard volume thresholds:
1. **Historical Search Volume**: $\text{GSC Impressions}_{\text{Feb-Apr}} \ge 1,000$
2. **Baseline Search Traffic**: $\text{GSC Clicks}_{\text{Apr}} \ge 10$

### Public Safety & Anonymization
Client and content identifiers are pseudonymized identifiers. They are used for grouping, joins, and deterministic ranking only. Zero raw URLs, private search queries, or client domain names appear anywhere in the analysis.

In [ ]:
# -------------------------------------------------------------------------
# Step 1: Secure Data Ingestion from DuckDB Warehouse
# -------------------------------------------------------------------------
token = os.getenv("HF_TOKEN")
if not token:
    try:
        token = getpass("Enter your Hugging Face READ token: ")
    except Exception:
        token = None

if not token:
    raise ValueError("HF_TOKEN environment variable not set. Please provide a valid Hugging Face token.")

con = duckdb.connect()
con.execute(f"CREATE SECRET IF NOT EXISTS hf_s (TYPE huggingface, TOKEN '{token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

w05_sql = f"""
WITH daily_facts AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        COALESCE(gsc_clicks, 0) AS gsc_clicks,
        gsc_avg_position,
        COALESCE(ga4_pageviews, 0) AS ga4_pageviews,
        COALESCE(ga4_engaged_sessions, 0) AS ga4_engaged_sessions,
        COALESCE(client_has_gsc, FALSE) AS client_has_gsc,
        COALESCE(client_has_ga4, FALSE) AS client_has_ga4,
        COALESCE(gsc_data_available, FALSE) AS gsc_data_available,
        COALESCE(ga4_data_available, FALSE) AS ga4_data_available
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month IN ('2026-02', '2026-03', '2026-04', '2026-05')
),
aggregated_pages AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature window aggregations (Feb - Apr 2026)
        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN gsc_impressions ELSE 0 END) AS gsc_impressions_feb_apr,
        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN gsc_clicks ELSE 0 END) AS gsc_clicks_feb_apr,
        SUM(CASE WHEN month = '2026-04' THEN gsc_impressions ELSE 0 END) AS gsc_impressions_apr,
        SUM(CASE WHEN month = '2026-04' THEN gsc_clicks ELSE 0 END) AS gsc_clicks_apr,
        AVG(CASE WHEN month = '2026-04' THEN gsc_avg_position ELSE NULL END) AS gsc_avg_position_apr,

        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN ga4_pageviews ELSE 0 END) AS ga4_pageviews_feb_apr,
        SUM(CASE WHEN month = '2026-04' THEN ga4_pageviews ELSE 0 END) AS ga4_pageviews_apr,
        SUM(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions_feb_apr,

        -- Bounded pre-May metadata & data availability flags
        MAX(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN client_has_gsc END) AS client_has_gsc,
        MAX(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN client_has_ga4 END) AS client_has_ga4,
        MAX(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN gsc_data_available END) AS gsc_data_available,
        MAX(CASE WHEN month IN ('2026-02', '2026-03', '2026-04') THEN ga4_data_available END) AS ga4_data_available,

        -- Target window aggregation (May 2026) - ISOLATED FOR TARGET ONLY
        SUM(CASE WHEN month = '2026-05' THEN gsc_clicks ELSE 0 END) AS gsc_clicks_may

    FROM daily_facts
    GROUP BY client_hash_id, content_hash_id
)
SELECT *
FROM aggregated_pages
WHERE gsc_impressions_feb_apr >= 1000
  AND gsc_clicks_apr >= 10
"""

df = con.sql(w05_sql).df()
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

# Compute Week 4 baseline score (daily average across April)
baseline_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(
        (
            CASE WHEN COALESCE(gsc_impressions,0) >= 10 THEN 1 ELSE 0 END +
            CASE WHEN gsc_avg_position > 10 THEN 1 ELSE 0 END +
            CASE WHEN COALESCE(ga4_pageviews,0) >= 1 THEN 1 ELSE 0 END
        )
    ) AS baseline_score
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-04'
GROUP BY
    client_hash_id,
    content_hash_id
"""
baseline_df = con.sql(baseline_sql).df()
df = df.merge(baseline_df, on=["client_hash_id", "content_hash_id"], how="left")
df["baseline_score"] = df["baseline_score"].fillna(0.0)

# Target definition (May 2026 decay >= 20%)
df["is_declining_label"] = (
    df["gsc_clicks_may"] < (0.80 * df["gsc_clicks_apr"])
).astype(int)
df["gsc_avg_position_apr"] = df["gsc_avg_position_apr"].fillna(0.0)

# Feature engineering (W05 official features)
df["log_gsc_impressions_feb_apr"] = np.log1p(df["gsc_impressions_feb_apr"])
df["log_gsc_clicks_apr"] = np.log1p(df["gsc_clicks_apr"])
df["log_ga4_pageviews_feb_apr"] = np.log1p(df["ga4_pageviews_feb_apr"])
df["log_ga4_engaged_sessions_feb_apr"] = np.log1p(df["ga4_engaged_sessions_feb_apr"])

feature_cols = [
    "log_gsc_impressions_feb_apr",
    "log_gsc_clicks_apr",
    "gsc_avg_position_apr",
    "log_ga4_pageviews_feb_apr",
    "log_ga4_engaged_sessions_feb_apr",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
]

X = df[feature_cols].astype(float)
y = df["is_declining_label"].astype(int)
groups = df["client_hash_id"].astype(str)

print(f"Cohort Summary: N = {len(df):,} eligible pages across {groups.nunique()} clients.")
print(f"Target Class Balance: {y.value_counts().to_dict()} (Overall Base Rate: {y.mean():.4f})")

## 3. Methodology & Validation Design

### Supervised Learning Setup
- **Classification Model**: Standardized `LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)` embedded in a scikit-learn `Pipeline` with `StandardScaler`.
- **9 Production Features**: 5 continuous historical/static search and analytics metrics, 4 of them log-transformed, + 4 boolean platform availability flags.
- **Target Formulation**: Binary decline indicator $y \in \{0, 1\}$, where $y=1$ if $\text{Clicks}_{\text{May}} < 0.80 \times \text{Clicks}_{\text{Apr}}$.

### Heuristic Baseline Formulation
For each page, the baseline computes the mean daily score across its available April records, where each daily score is the sum of three binary signals: GSC impressions $\ge 10$, average position $> 10$, and GA4 pageviews $\ge 1$:
$$\text{Daily Score}_d = \mathbf{1}_{\{\text{gsc\_impressions} \ge 10\}} + \mathbf{1}_{\{\text{gsc\_avg\_position} > 10\}} + \mathbf{1}_{\{\text{ga4\_pageviews} \ge 1\}}$$
$$\text{Baseline Score} = \text{mean of Daily Score across available April records}$$

### Validation Design (Grouped Holdout Split)
To reduce leakage from client-specific patterns, we split pages strictly by client using `GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)`:
- **Training Split**: 28 clients (14,786 pages)
- **Held-Out Test Split**: 8 clients (1,727 pages)

### Leakage Check
All nine model features are constructed from data available through April 30, 2026, while the decline label uses May 2026 clicks. Client and content identifiers are used only for grouping, joins, and deterministic ranking, not as predictive features. The ML-09 validation audit also performed a controlled future-data injection using May clicks. The intentionally leaky version produced a large increase in Precision@50 compared with the clean model (clean W05 P@50 = 0.2400 vs. leaky May-click-injected P@50 = 0.9800), demonstrating how future information can materially distort evaluation. The full leakage audit is documented in `w06_validation_audit.ipynb`.

### Evaluation Metric (Precision@50)
Because editorial teams review a fixed queue, our primary metric is **Precision@50** (the proportion of true declining pages in the top 50 ranked candidates). Deterministic tie-breaking orders pages by: `Score DESC → April Impressions DESC → April Pageviews DESC → Position ASC`.

In [ ]:
# Evaluation helper functions
def precision_at_k(y_true, scores, tie_df, k=50):
    eval_df = tie_df.copy()
    eval_df["y"] = list(y_true)
    eval_df["score"] = list(scores)
    top = eval_df.sort_values(
        ["score", "gsc_impressions_apr", "ga4_pageviews_apr", "gsc_avg_position_apr"],
        ascending=[False, False, False, True]
    ).head(k)
    return float(top["y"].mean()), int(top["y"].sum())

def baseline_precision_at_k(test_subset, k=50):
    top = test_subset.sort_values(
        ["baseline_score", "gsc_impressions_apr", "ga4_pageviews_apr", "gsc_avg_position_apr"],
        ascending=[False, False, False, True]
    ).head(k)
    return float(top["is_declining_label"].mean()), int(top["is_declining_label"].sum())

# Primary W05 Client-Grouped Split (seed=42)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))

X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
test_df = df.iloc[te_idx].copy()

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))
])
lr_pipeline.fit(X_tr, y_tr)
test_probs = lr_pipeline.predict_proba(X_te)[:, 1]

print(f"Training Set: {len(X_tr):,} rows across {groups.iloc[tr_idx].nunique()} clients")
print(f"Held-Out Test Set: {len(X_te):,} rows across {groups.iloc[te_idx].nunique()} clients")

## 4. Results (Model vs. Baseline & Stability Audit)

### Primary Held-Out Split Evaluation
Evaluated on the exact same held-out test pages ($N=1,727$ across 8 clients, test base rate = $38.7\%$):
- **Week 4 Heuristic Baseline**: Precision@50 = **`0.2000`** ($10/50$ true declining pages).
- **Logistic Regression Model**: Precision@50 = **`0.2400`** ($12/50$ true declining pages).
- **Single-Split Measured Gain**: **`+0.0400`** ($+4.0$ percentage points).

---

### Enhanced Validation Audit: 5 Repeated Client-Grouped Splits
To assess how sensitive the observed +4.0 pp lift is to the client partition, we run 5 repeated GroupShuffleSplit evaluations (test_size=0.20, seeds 42, 43, 44, 45, 46):

| Split Seed | Held-Out Pages | Held-Out Clients | Baseline P@50 | Logistic Regression P@50 | Split Lift ($\Delta$) |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **Seed 42** | 1,727 | 8 | 0.2000 (10/50) | 0.2400 (12/50) | +0.0400 (+4.0 pp) |
| **Seed 43** | 6,324 | 8 | 0.1400 (7/50) | 0.2800 (14/50) | +0.1400 (+14.0 pp) |
| **Seed 44** | 4,763 | 8 | 0.5600 (28/50) | 0.4400 (22/50) | -0.1200 (-12.0 pp) |
| **Seed 45** | 802 | 8 | 0.5600 (28/50) | 0.4800 (24/50) | -0.0800 (-8.0 pp) |
| **Seed 46** | 3,796 | 8 | 0.1600 (8/50) | 0.2600 (13/50) | +0.1000 (+10.0 pp) |

- **Baseline 5-Split Mean**: **`0.3240 ± 0.2165`** (Range: `0.1400` to `0.5600`)
- **Logistic Regression 5-Split Mean**: **`0.3400 ± 0.1114`** (Range: `0.2400` to `0.4800`)
- **Mean Split Lift**: **`+0.0160 ± 0.1126`** ($+1.6\text{ pp}$, Range: $-12.0\text{ pp}$ to $+14.0\text{ pp}$)

> **Key Empirical Finding**: Logistic Regression outperformed the baseline in 3 of 5 splits and underperformed in 2, with split-level lift ranging from -12.0 to +14.0 percentage points. The original +4.0 pp result should therefore be treated as a measured result for the W05 partition rather than a stable improvement across client portfolios.

In [ ]:
# 1. Primary Split Evaluation
orig_base_p50, orig_base_tp = baseline_precision_at_k(test_df, k=50)
orig_lr_p50, orig_lr_tp = precision_at_k(y_te, test_probs, test_df, k=50)

print("=== Primary W05 Client-Grouped Evaluation (Seed 42) ===")
print(f"Week 4 Baseline Rule Precision@50: {orig_base_p50:.4f} ({orig_base_tp}/50)")
print(f"Logistic Regression  Precision@50: {orig_lr_p50:.4f} ({orig_lr_tp}/50)")
print(f"Measured Single-Split Lift:        {orig_lr_p50 - orig_base_p50:+.4f}")

# 2. Repeated 5-Split Evaluation (Numeric Lift)
split_seeds = [42, 43, 44, 45, 46]
split_records = []

for seed in split_seeds:
    s_gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    s_tr_idx, s_te_idx = next(s_gss.split(X, y, groups=groups))
    
    s_X_tr, s_X_te = X.iloc[s_tr_idx], X.iloc[s_te_idx]
    s_y_tr, s_y_te = y.iloc[s_tr_idx], y.iloc[s_te_idx]
    s_test_df = df.iloc[s_te_idx].copy()
    
    s_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000))
    ])
    s_pipeline.fit(s_X_tr, s_y_tr)
    s_test_probs = s_pipeline.predict_proba(s_X_te)[:, 1]
    
    s_base_p50, s_base_tp = baseline_precision_at_k(s_test_df, k=50)
    s_lr_p50, s_lr_tp = precision_at_k(s_y_te, s_test_probs, s_test_df, k=50)
    
    split_records.append({
        "Seed": seed,
        "Held-Out Pages": len(s_test_df),
        "Held-Out Clients": groups.iloc[s_te_idx].nunique(),
        "Baseline P@50": s_base_p50,
        "Baseline TP": f"{s_base_tp}/50",
        "LR P@50": s_lr_p50,
        "LR TP": f"{s_lr_tp}/50",
        "Lift": float(s_lr_p50 - s_base_p50)
    })

split_df = pd.DataFrame(split_records)
print("\n=== Repeated 5-Split GroupShuffleSplit Validation Summary ===")
display(split_df)

baseline_mean = split_df["Baseline P@50"].mean()
baseline_std = split_df["Baseline P@50"].std()
lr_mean = split_df["LR P@50"].mean()
lr_std = split_df["LR P@50"].std()
lift_mean = split_df["Lift"].mean()
lift_std = split_df["Lift"].std()

print(f"\nComputed 5-Split Baseline: {baseline_mean:.4f} ± {baseline_std:.4f} (Range: {split_df['Baseline P@50'].min():.4f} to {split_df['Baseline P@50'].max():.4f})")
print(f"Computed 5-Split LR:       {lr_mean:.4f} ± {lr_std:.4f} (Range: {split_df['LR P@50'].min():.4f} to {split_df['LR P@50'].max():.4f})")
print(f"Computed 5-Split Lift:     {lift_mean:+.4f} ± {lift_std:.4f} (Range: {split_df['Lift'].min():+.4f} to {split_df['Lift'].max():+.4f})")

# 3. Standardized Feature Coefficients
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Standardized Coefficient": lr_pipeline.named_steps["model"].coef_[0]
}).sort_values("Standardized Coefficient", key=abs, ascending=False).reset_index(drop=True)

print("\n=== Standardized Logistic Regression Feature Coefficients ===")
display(coef_df)

## 5. Limitations & Honest Claim Framing

### Methodological Boundaries & Negative Findings
1. **Observational Association $\neq$ Causal Intervention Lift**: The model flags statistical vulnerabilities based on historical search and analytics data. It does not establish that executing a specific content edit will causally reverse decay or increase rankings.
2. **Client Partition Sensitivity**: Across five repeated client-grouped holdouts, model lift ranged from -12.0 to +14.0 percentage points. The single-split $+4.0\text{ pp}$ improvement is a measured result for that partition, not a universal guarantee.
3. **In-Sample Queue Disclosure**: When scoring the entire 16,513-page portfolio for decision-support queue generation, pages from the training split receive in-sample scores. Out-of-sample expectations must trace strictly to the client-holdout validation design.
4. **No Search Algorithm Reverse-Engineering**: The model does not predict or reverse-engineer Google's proprietary search algorithms. It models statistical associations between historical search/engagement signals and the subsequent May click-decline label within this specific enterprise portfolio.
5. **High False-Positive Rate in Top Queue**: At Precision@50 = 0.2400, 38 out of 50 queued pages do not experience $\ge 20\%$ decay. False positives are pages in the top-50 model queue that did not meet the study's May decline label. The tool functions strictly as candidate triage to direct human diagnostic review.

In [ ]:
# Held-Out Top-50 Error Analysis
pred_df = test_df.copy()
pred_df["predicted_decline_prob"] = test_probs
top50 = pred_df.sort_values(
    ["predicted_decline_prob", "gsc_impressions_apr", "ga4_pageviews_apr", "gsc_avg_position_apr"],
    ascending=[False, False, False, True]
).head(50)

tp_count = int(top50["is_declining_label"].sum())
fp_count = int(len(top50) - tp_count)

print("=== Held-Out Test Queue Error Breakdown ===")
print(f"Total Reviewed Candidates: {len(top50)}")
print(
    f"True Positives (Pages Meeting the Decline Label): "
    f"{tp_count} ({tp_count/len(top50)*100:.1f}%)"
)
print(f"False Positives (Pages Not Meeting the Decline Label): {fp_count} ({fp_count/len(top50)*100:.1f}%)")

## 6. Ranked Recommendations & Action Playbook

### The 5 Content Archetypes & Human Review Workflows
To operationalize predictions without over-relying on automated interventions, we categorize pages into **5 illustrative review archetypes**:

1. **Striking Distance / Search Visibility** ($8.0 \le \text{Position} \le 20.0$, Impressions $\ge 500$):
   - **Recommended Review**: Review title/meta alignment, search intent, SERP competition, and page structure.
   - **Illustrative Effort Assumption**: ~2.0 hours.

2. **High Historical Traffic / Higher Decline Risk** (GA4 Pageviews $\ge 100$, Decline Prob $\ge 0.45$):
   - **Recommended Review**: Review for substantive content freshness, factual accuracy, intent alignment, and technical issues.
   - **Illustrative Effort Assumption**: ~5.0 hours.

3. **High Search Impression / Thin Traffic** (Impressions $\ge 1000$, GA4 Pageviews $< 50$):
   - **Recommended Review**: Review click-through performance, search intent alignment, content depth, and internal linking.
   - **Illustrative Effort Assumption**: ~3.5 hours.

4. **General Higher-Decline-Risk Review** (Decline Prob $\ge 0.50$):
   - **Recommended Review**: Prioritize for diagnostic review of recent traffic trends and on-page content.
   - **Illustrative Effort Assumption**: ~3.0 hours.

5. **Lower-Priority / No Specific Review Signal** (No specific review trigger):
   - **Recommended Action**: Lower-priority monitoring.
   - **Illustrative Effort Assumption**: ~0.5 hours.

### Strict Human Review & The No-Go List
- ❌ **No Autonomous LLM Overwriting**: Never deploy unattended AI generation directly to production URLs without human domain review.
- ❌ **No Programmatic URL Redirects / Canonicals**: Model scores never authorize URL alterations or 301 redirects.
- ❌ **No Automated Page Pruning or Deletion**: High-value legacy URLs must never be pruned programmatically.
- ❌ **No Automated Updates to YMYL / High-Stakes Content**: Legal, medical, or financial advice URLs require mandatory human compliance signoff.

In [ ]:
# Score entire eligible portfolio for decision-support queue generation
df["predicted_decline_prob"] = lr_pipeline.predict_proba(X)[:, 1]
df["apr_ctr"] = np.where(df["gsc_impressions_apr"] > 0, df["gsc_clicks_apr"] / df["gsc_impressions_apr"], 0.0)

def assign_archetype_and_action(row):
    prob = row["predicted_decline_prob"]
    pos = row["gsc_avg_position_apr"]
    impr = row["gsc_impressions_apr"]
    pv = row["ga4_pageviews_feb_apr"]
    ctr = row["apr_ctr"]
    
    reasons = []
    if prob >= 0.50:
        reasons.append("HIGH_MODEL_DECLINE_RISK")
    if pos >= 8.0 and pos <= 20.0:
        reasons.append("STRIKING_DISTANCE_POS_8_20")
    if ctr < 0.015 and impr >= 500:
        reasons.append("LOW_CTR_HIGH_SEARCH_IMPR")
    if pv >= 100 and prob >= 0.45:
        reasons.append("HIGH_HISTORICAL_TRAFFIC_MODEL_RISK")
    if pv < 20 and impr >= 1000:
        reasons.append("THIN_PAGEVIEWS_HIGH_SEARCH_DEMAND")
    if row["baseline_score"] >= 2.0:
        reasons.append("HIGH_W04_BASELINE_SIGNAL")
        
    reason_str = " | ".join(reasons) if reasons else "ROUTINE_MONITORING"
    
    # Deterministic Archetype Assignment (5 Archetypes)
    if pos >= 8.0 and pos <= 20.0 and impr >= 500:
        archetype = "Striking Distance / Search Visibility"
        action = "Review title/meta alignment, search intent, SERP competition, and page structure."
        effort_hrs = 2.0
    elif pv >= 100 and prob >= 0.45:
        archetype = "High Historical Traffic / Higher Decline Risk"
        action = "Review for substantive content freshness, factual accuracy, intent alignment, and technical issues."
        effort_hrs = 5.0
    elif impr >= 1000 and pv < 50:
        archetype = "High Search Impression / Thin Traffic"
        action = "Review click-through performance, search intent alignment, content depth, and internal linking."
        effort_hrs = 3.5
    elif prob >= 0.50:
        archetype = "General Higher-Decline-Risk Review"
        action = "Prioritize for diagnostic review of recent traffic trends and on-page content."
        effort_hrs = 3.0
    else:
        archetype = "Lower-Priority / No Specific Review Signal"
        action = "Lower-priority monitoring."
        effort_hrs = 0.5
        
    if prob >= 0.60:
        priority = "P1_High_Priority"
    elif prob >= 0.45:
        priority = "P2_Review"
    else:
        priority = "P3_Monitor"
        
    guardrails = ["MODEL_SCORE_DOES_NOT_AUTHORIZE_REDIRECT_OR_CANONICAL_CHANGE"]
    if prob >= 0.70:
        guardrails.append("MANDATORY_SENIOR_EDITOR_SIGNOFF")
    safety_guardrail = " | ".join(guardrails)
    
    return pd.Series(
        [archetype, action, effort_hrs, reason_str, priority, safety_guardrail],
        index=["content_archetype", "recommended_action", "effort_hours_assumption", "reason_codes", "priority_tier", "safety_guardrail"]
    )

df[["content_archetype", "recommended_action", "effort_hours_assumption", "reason_codes", "priority_tier", "safety_guardrail"]] = (
    df.apply(assign_archetype_and_action, axis=1)
)

ranked_queue = df.sort_values(
    ["predicted_decline_prob", "gsc_impressions_apr", "ga4_pageviews_apr", "gsc_avg_position_apr"],
    ascending=[False, False, False, True]
).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

print("=== Decision-Support Playbook: Top 10 Ranked Candidates ===")
display(ranked_queue[["rank", "content_hash_id", "predicted_decline_prob", "content_archetype", "priority_tier", "effort_hours_assumption", "safety_guardrail"]].head(10))

## 7. Artifacts & Figures the Paper Embeds

### Exporting Deliverables to `work/outputs/` and `work/figures/`
We generate and persist all data tables, metric JSONs, and paper-ready figures required for our static research paper:
- `work/outputs/content_action_playbook_queue.csv`: Full portfolio ranked queue (gitignored data file).
- `work/outputs/playbook_summary_metrics.json`: Summary receipts and tier counts.
- `work/figures/action_archetype_distribution.png`: Paper-ready horizontal stacked bar chart.
- `work/figures/effort_vs_risk_matrix.png`: Paper-ready scatter plot of decline risk vs. effort assumptions.

In [ ]:
output_dir = "work/outputs"
figure_dir = "work/figures"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(figure_dir, exist_ok=True)

# 1. Export CSV Queue
queue_export_cols = [
    "rank", "client_hash_id", "content_hash_id", "predicted_decline_prob",
    "baseline_score", "content_archetype", "recommended_action",
    "priority_tier", "effort_hours_assumption", "safety_guardrail",
    "gsc_impressions_apr", "gsc_clicks_apr", "gsc_avg_position_apr",
    "ga4_pageviews_apr", "reason_codes"
]
csv_path = os.path.join(output_dir, "content_action_playbook_queue.csv")
ranked_queue[queue_export_cols].to_csv(csv_path, index=False)
print(f"[Export] Saved Queue CSV -> {csv_path} ({len(ranked_queue):,} rows)")

# 2. Export Metrics JSON (Matching current final values)
summary_metrics = {
    "dataset": {
        "total_eligible_pages": int(len(ranked_queue)),
        "total_clients": int(ranked_queue["client_hash_id"].nunique()),
        "mean_portfolio_predicted_decline_risk": float(ranked_queue["predicted_decline_prob"].mean())
    },
    "validation_summary": {
        "w05_baseline_p50": float(orig_base_p50),
        "w05_lr_p50": float(orig_lr_p50),
        "w05_lift": float(orig_lr_p50 - orig_base_p50),
        "repeated_5split_baseline_mean": float(baseline_mean),
        "repeated_5split_baseline_std": float(baseline_std),
        "repeated_5split_lr_mean": float(lr_mean),
        "repeated_5split_lr_std": float(lr_std),
        "repeated_5split_lift_mean": float(lift_mean),
        "repeated_5split_lift_std": float(lift_std)
    },
    "priority_tier_counts": ranked_queue["priority_tier"].value_counts().to_dict(),
    "archetype_counts": ranked_queue["content_archetype"].value_counts().to_dict(),
    "recommended_action_counts": ranked_queue["recommended_action"].value_counts().to_dict(),
    "safety_guardrail_counts": ranked_queue["safety_guardrail"].value_counts().to_dict(),
    "editorial_bandwidth_assumptions": {
        "top_50_queue_effort_hours": float(ranked_queue.head(50)["effort_hours_assumption"].sum()),
        "p1_high_priority_effort_hours": float(ranked_queue[ranked_queue["priority_tier"] == "P1_High_Priority"]["effort_hours_assumption"].sum()),
        "total_portfolio_effort_hours": float(ranked_queue["effort_hours_assumption"].sum())
    }
}
json_path = os.path.join(output_dir, "playbook_summary_metrics.json")
with open(json_path, "w") as f:
    json.dump(summary_metrics, f, indent=2)
print(f"[Export] Saved Metrics JSON -> {json_path}")

# 3. Export Figure 1: Action Distribution
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig1, ax1 = plt.subplots(figsize=(10, 5), dpi=300)
tier_action_df = pd.crosstab(
    ranked_queue["recommended_action"],
    ranked_queue["priority_tier"]
)[["P1_High_Priority", "P2_Review", "P3_Monitor"]]

tier_action_df.plot(
    kind="barh",
    stacked=True,
    color=["#D95F02", "#7570B3", "#1B9E77"],
    ax=ax1,
    edgecolor="black",
    alpha=0.88
)
ax1.set_title("Content Action Playbook: Recommended Actions by Priority Tier", fontsize=13, fontweight="bold", pad=12)
ax1.set_xlabel("Number of Eligible Content Pages", fontsize=11, labelpad=8)
ax1.set_ylabel("Recommended Review Workflow", fontsize=11, labelpad=8)
ax1.legend(title="Priority Tier", frameon=True, facecolor="white", edgecolor="gray")
plt.tight_layout()
fig1_path = os.path.join(figure_dir, "action_archetype_distribution.png")
plt.savefig(fig1_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[Export] Saved Figure 1 -> {fig1_path}")

# 4. Export Figure 2: Effort vs Risk Matrix
fig2, ax2 = plt.subplots(figsize=(9, 5.5), dpi=300)
palette = {"P1_High_Priority": "#D95F02", "P2_Review": "#7570B3", "P3_Monitor": "#1B9E77"}
sample_plot_df = ranked_queue.sample(min(2000, len(ranked_queue)), random_state=42)

sns.scatterplot(
    data=sample_plot_df,
    x="effort_hours_assumption",
    y="predicted_decline_prob",
    hue="priority_tier",
    palette=palette,
    alpha=0.6,
    s=40,
    ax=ax2,
    edgecolor="none"
)
ax2.axhline(0.50, color="red", linestyle="--", linewidth=1.2, alpha=0.7, label="Illustrative higher-priority heuristic cutoff (P >= 0.50)")
ax2.set_title("Decision-Support Matrix: Predicted Decline Probability vs. Effort Assumption", fontsize=13, fontweight="bold", pad=12)
ax2.set_xlabel("Illustrative Editorial Effort Assumption (Hours)", fontsize=11, labelpad=8)
ax2.set_ylabel("Model Predicted Decline Probability", fontsize=11, labelpad=8)
ax2.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="gray", fontsize=9)
plt.tight_layout()
fig2_path = os.path.join(figure_dir, "effort_vs_risk_matrix.png")
plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[Export] Saved Figure 2 -> {fig2_path}")

## 8. Reproducibility & Data Credit

### Reproducibility Guide
Reproduction of the warehouse-backed analysis requires authorized access to the FlyRank Internship dataset.
To re-run this analysis pipeline from a cloned environment with valid credentials:
```bash
git clone https://github.com/ShubhamSnSharma/flyrank-ml-internship.git
cd flyrank-ml-internship
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
export HF_TOKEN="your_huggingface_read_token"
# Run the notebook directly or inspect outputs in work/outputs/
```

### Acknowledgments & Data Credit
> **Data Attribution**: This research was built on the **FlyRank ML Internship dataset**, provided by [FlyRank.ai](https://flyrank.ai/). We thank the FlyRank team for providing research access to the 79-million-row search performance warehouse.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.